# uv 与 Python 环境管理

## 为什么需要 uv？

- **pip**：安装第三方包（别人写好发布到 PyPI——Python 官方的包仓库——上的代码库）。
- **venv / virtualenv**：创建虚拟环境。
- **pyenv**：安装和切换不同 Python 版本。
- **pip-tools**：锁定精确依赖版本。`requirements.txt` 只能记顶层依赖，锁不准。
- **pipx**：安装命令行工具，让它们在 PATH 里可用但不污染全局 Python。

**uv可以做到这5项**

## uv 是什么？

**uv** 是一个用 Rust 写的**Python 项目管理工具**。

```
pyproject.toml（项目配置：项目名、Python版本、依赖列表）
    ↓
uv（读 pyproject.toml，一站式搞定）
    ├── uv init          → 初始化项目
    ├── uv venv          → 创建虚拟环境
    ├── uv add / uv pip  → 安装依赖包
    ├── uv lock          → 锁定精确版本
    └── uv sync          → 一键同步整个环境
```

**poetry** 可以做到全生命周期：依赖+构建+打包+发布

**conda**跨语言包管理器，连 C/Fortran/CUDA 等非 Python 底层依赖一起管

## 怎么用？

### 安装 uv

In [ ]:
# Windows（PowerShell）
powershell -c "irm https://astral.sh/uv/install.ps1 | iex"

# macOS / Linux
curl -LsSf https://astral.sh/uv/install.sh | sh

# 通用（需要已有 Python）
pip install uv

# 验证安装
uv --version

### 使用uv

In [ ]:
uv init my-project         # 创建项目文件夹，自动生成 pyproject.toml
uv init --app              # 面向应用（默认）
uv init --lib              # 面向库（写给别人用的 python 依赖，会生成构建配置）
uv init --no-readme        # 不生成 README.md

### uv add / uv remove：管理依赖

In [ ]:
uv add flask                # 安装 flask，写入 pyproject.toml
uv add "requests>=2.28"     # 指定版本范围
uv add pytest --dev         # 安装为开发依赖（只开发时需要）
uv add "black ruff" --dev   # 一次装 black ruff 多个开发工具

uv remove flask             # 卸载并自动从 pyproject.toml 删除

> 生产依赖是项目运行必须的；开发依赖只在你开发、测试、格式化代码时才需要。

### uv lock 与 uv sync：锁定和同步

lock **把版本范围「锁」成精确版本**。

In [ ]:
uv lock                       # 解析依赖 → 生成 uv.lock
uv sync                       # 按 uv.lock 安装所有依赖（别人拿到项目也跑这条）
uv sync --no-dev              # 只装生产依赖，不装开发工具

# 如果 pyproject.toml 变了（加了/删了依赖），重新锁 + 同步：
uv lock && uv sync

### uv venv：创建虚拟环境

In [ ]:
uv venv                          # 在当前目录创建 .venv 文件夹
uv venv --python 3.12            # 指定 Python 版本（uv 会自动下载）
uv venv myenv                    # 指定虚拟环境名称（默认 .venv）

# 激活虚拟环境
# uv run 会绕过 PATH，直接去 .venv/ 里找 python 执行

> 激活虚拟环境会修改**当前终端**的 PATH 环境变量，把 `.venv/bin` 插到最前面。这样在这个终端里敲 `python`，系统会优先找到 `.venv/` 里的版本，而不是系统全局的，关掉终端就失效。

### uv pip：兼容传统 pip 用法

如果你习惯了 pip 的命令，uv 提供了完全兼容的接口（只是更快）：

In [ ]:
uv pip install flask          # 等同于 pip install flask
uv pip install -r requirements.txt   # 从 requirements.txt 安装
uv pip list                   # 列出当前环境所有已安装的包
uv pip uninstall flask        # 卸载
uv pip freeze > requirements.txt     # 导出依赖列表

> `requirements.txt` 是 Python 项目记录依赖的传统方式，每行一个包名。但它只能记**顶层依赖**，不能锁定**间接依赖**的版本。

### uv python：管理 Python 版本

uv 可以下载和管理多个 Python 版本。

In [ ]:
uv python list                # 查看 uv 已安装的 Python 版本
uv python install 3.12        # 下载 Python 3.12
uv python install 3.11 3.10   # 同时下载多个版本
uv venv --python 3.12         # 用指定版本创建虚拟环境

## 典型工作流

从零开始一个 Python 项目，使用 uv 的最简流程：

In [ ]:
# 1. 创建项目（自动生成 pyproject.toml、README、.gitignore）
uv init my-project
cd my-project

# 2. 装依赖（uv add/sync 自动创建 venv + 生成 uv.lock + 自动装 python 版本）
uv add flask
uv add pytest --dev

# 3. 别人拿到项目，一键同步
uv sync

# 4. 开发……（不激活也能跑）
uv run python main.py

---

## 本篇出现的新名词速查

| 名词 | 一句话解释 |
|------|-----------|
| uv | Rust 写的超快 Python 包管理器，pip+venv+pyenv+pip-tools+pipx 五合一 |
| pip | Python 的传统包安装工具，从 PyPI 下载并安装第三方包 |
| PyPI | Python Package Index，Python 的官方第三方包仓库（`pypi.org`） |
| 虚拟环境 (venv) | `.venv/` 文件夹里的独立 Python 小世界，安装的包只对当前项目生效 |
| 依赖 (Dependency) | 你的代码需要借用的别人写的库 |
| 间接依赖 | 你的依赖的依赖——flask 依赖 jinja2，jinja2 就是间接依赖 |
| pyproject.toml | Python 项目的标准配置文件（PEP 621），声明项目名、版本、依赖列表 |
| uv.lock | 锁文件，记录每个依赖的精确版本和哈希值，保证团队环境一致 |
| uv add | 安装包并自动写入 pyproject.toml（≈ pip install + 手动编辑配置文件） |
| uv sync | 一键同步：创建 venv + 按 lock 文件安装所有依赖 |
| requirements.txt | 传统依赖记录方式，每行一个包名，简单但功能有限 |
| curl | 在终端里发网络请求的命令行工具 |
| 激活 (activate) | 把终端 PATH 指向虚拟环境里的 Python，使其在当前终端生效 |
| Rust | 一种注重性能和内存安全的系统编程语言，uv 用 Rust 写所以极快 |
| conda | 跨语言包管理器 + 环境管理器，pip 只管 Python 包，conda 连 C/CUDA 库一起管 |
| Poetry | uv 出现之前的现代 Python 项目管理工具，管依赖+构建+打包+发布全生命周期，uv 正在快速取代它 |
| Anaconda | conda 的全家桶发行版，预装 150+ 科学库，体积巨大 |